In [1]:
import os
import shutil
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, DateType
from pyspark.sql import Row
from pyspark.sql.functions import expr
from pyspark.sql.functions import *

spark = (
    SparkSession.builder
    .appName("Spark Recap")
    .master("local[2]")
    .getOrCreate()
)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/09 01:55:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/09 01:55:47 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/09 01:55:47 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [4]:
carsDF = spark.read.json("src/main/resources/data/cars")


carsDF.show(5)

+------------+---------+------------+----------+----------------+--------------------+------+-------------+----------+
|Acceleration|Cylinders|Displacement|Horsepower|Miles_per_Gallon|                Name|Origin|Weight_in_lbs|      Year|
+------------+---------+------------+----------+----------------+--------------------+------+-------------+----------+
|        12.0|        8|       307.0|       130|            18.0|chevrolet chevell...|   USA|         3504|1970-01-01|
|        11.5|        8|       350.0|       165|            15.0|   buick skylark 320|   USA|         3693|1970-01-01|
|        11.0|        8|       318.0|       150|            18.0|  plymouth satellite|   USA|         3436|1970-01-01|
|        12.0|        8|       304.0|       150|            16.0|       amc rebel sst|   USA|         3433|1970-01-01|
|        10.5|        8|       302.0|       140|            17.0|         ford torino|   USA|         3449|1970-01-01|
+------------+---------+------------+----------+

In [6]:
# select
usefulCarsData = (
    carsDF.select(col("Name"), 
                  carsDF["Year"], 
                  (col("Weight_in_lbs") / 2.2).alias("Weight_in_kg"),
                  expr("Weight_in_lbs / 2.2").alias("Weight_in_kg_2"))
)

usefulCarsData.show(5)

+--------------------+----------+------------------+--------------+
|                Name|      Year|      Weight_in_kg|Weight_in_kg_2|
+--------------------+----------+------------------+--------------+
|chevrolet chevell...|1970-01-01|1592.7272727272725|   1592.727273|
|   buick skylark 320|1970-01-01|1678.6363636363635|   1678.636364|
|  plymouth satellite|1970-01-01|1561.8181818181818|   1561.818182|
|       amc rebel sst|1970-01-01|1560.4545454545453|   1560.454545|
|         ford torino|1970-01-01|1567.7272727272725|   1567.727273|
+--------------------+----------+------------------+--------------+
only showing top 5 rows



In [7]:
# select expr
carsDF.selectExpr("Weight_in_lbs / 2.2").show(5)

+---------------------+
|(Weight_in_lbs / 2.2)|
+---------------------+
|          1592.727273|
|          1678.636364|
|          1561.818182|
|          1560.454545|
|          1567.727273|
+---------------------+
only showing top 5 rows



In [8]:
# filter
carsDF.filter(col("Origin") != "USA").show(5)

+------------+---------+------------+----------+----------------+--------------------+------+-------------+----------+
|Acceleration|Cylinders|Displacement|Horsepower|Miles_per_Gallon|                Name|Origin|Weight_in_lbs|      Year|
+------------+---------+------------+----------+----------------+--------------------+------+-------------+----------+
|        17.5|        4|       133.0|       115|            NULL|citroen ds-21 pallas|Europe|         3090|1970-01-01|
|        15.0|        4|       113.0|        95|            24.0|toyota corona mar...| Japan|         2372|1970-01-01|
|        14.5|        4|        97.0|        88|            27.0|        datsun pl510| Japan|         2130|1970-01-01|
|        20.5|        4|        97.0|        46|            26.0|volkswagen 1131 d...|Europe|         1835|1970-01-01|
|        17.5|        4|       110.0|        87|            25.0|         peugeot 504|Europe|         2672|1970-01-01|
+------------+---------+------------+----------+

In [9]:
# aggregations
carsDF.select(avg(col("Horsepower")).alias("average_hp")).show() # sum, mean, stddev, min, max, etc.

+----------+
|average_hp|
+----------+
|  105.0825|
+----------+



In [10]:
# grouping
(
    carsDF.groupBy(col("Origin")) # relational grouped dataset
    .count()
).show()

+------+-----+
|Origin|count|
+------+-----+
|Europe|   73|
|   USA|  254|
| Japan|   79|
+------+-----+



In [11]:
# joining
guitarPlayers = spark.read.json("src/main/resources/data/guitarPlayers")
bands = spark.read.json("src/main/resources/data/bands")

guitarPlayers.show(5)
bands.show(5)

+----+-------+---+------------+
|band|guitars| id|        name|
+----+-------+---+------------+
|   0|    [0]|  0|  Jimmy Page|
|   1|    [1]|  1| Angus Young|
|   2| [1, 5]|  2|Eric Clapton|
|   3|    [3]|  3|Kirk Hammett|
+----+-------+---+------------+

+-----------+---+------------+----+
|   hometown| id|        name|year|
+-----------+---+------------+----+
|     Sydney|  1|       AC/DC|1973|
|     London|  0|Led Zeppelin|1968|
|Los Angeles|  3|   Metallica|1981|
|  Liverpool|  4| The Beatles|1960|
+-----------+---+------------+----+



In [ ]:
# join data frames
# join types
# inner - only matching rows are kept
# left/right outer join - all rows kept from one side with nulls for non matching
# antijoin - only show rows that dont match the other
# semijoin - join rows that exist but do not include the actual data from other table
guitaristsBands = guitarPlayers.join(bands, guitarPlayers["band"] == bands["id"])


guitaristsBands.show(5)

+----+-------+---+------------+-----------+---+------------+----+
|band|guitars| id|        name|   hometown| id|        name|year|
+----+-------+---+------------+-----------+---+------------+----+
|   1|    [1]|  1| Angus Young|     Sydney|  1|       AC/DC|1973|
|   0|    [0]|  0|  Jimmy Page|     London|  0|Led Zeppelin|1968|
|   3|    [3]|  3|Kirk Hammett|Los Angeles|  3|   Metallica|1981|
+----+-------+---+------------+-----------+---+------------+----+



In [18]:
# sql
carsDF.createOrReplaceTempView("cars")

spark.sql(
    """
    select Name from cars where Origin = 'USA'
    """
).show(5)

+--------------------+
|                Name|
+--------------------+
|chevrolet chevell...|
|   buick skylark 320|
|  plymouth satellite|
|       amc rebel sst|
|         ford torino|
+--------------------+
only showing top 5 rows



In [23]:
# low level: RDDs
# all spark jobs boil down to RDDs

sc = spark.sparkContext

numbersRDD = sc.parallelize([Row(num=x) for x in range(1000000)])

numbersRDD.count()

1000000

In [27]:
numbersRDD.map(lambda x: x.num * 2).count()

1000000

In [28]:
# rdd to data frame
numbersRDD.toDF().show(5)

+---+
|num|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+
only showing top 5 rows



In [ ]:
# data frame to rdd
carsRDD = carsDF.rdd

carsRDD.count()

406

26/03/31 11:00:54 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 31796396 ms exceeds timeout 120000 ms
26/03/31 11:00:54 WARN SparkContext: Killing executors is not supported by current scheduler.
26/03/31 11:01:03 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint